# Final Assignment: training

This notebook contains the source code for training the weights of the classification model.

## Download dataset

The following cell needs to be ran only once. It downloads and unzips the data. The unzip CLI tool might not be installed by default on Windows, so if you are using windows, please unzip manually into a folder called "data", or see the README.md.

In [1]:
# Downloading the dataset (unzip might not be installed on Windows)
# !curl -L -o ./data.zip https://www.kaggle.com/api/v1/datasets/download/navoneel/brain-mri-images-for-brain-tumor-detection
# !unzip -d ./data ./data.zip

## Importing libraries

See README.md for installing dependencies :).

In [4]:
import os
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torchvision import models, datasets, transforms
from tqdm import tqdm

## Setting the seed

Setting the seed for good reproducability :D

In [5]:
torch.manual_seed(21)  # Using my lucky number :D
np.random.seed(21)

## Setting the device used for training

In [6]:
device = "cpu"
if torch.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
print(f"Using: {device}")

Using: mps


## Preparing the dataset

In [7]:
transform = transforms.Compose(
    [
        transforms.Resize((299, 299)),  # 299 is the minimal image size of InceptionV3
        transforms.ToTensor()
    ]
)

In [8]:
path = os.path.join("data", "brain_tumor_dataset")

In [9]:
dataset = datasets.ImageFolder(
    root=path,
    transform=transform
)
train_dataset, test_dataset = random_split(
    dataset=dataset,
    lengths=[0.8, 0.2]
)

In [10]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

## Creating the model

### Loading pre-trained model

In [11]:
model = models.inception_v3(weights=models.Inception_V3_Weights)

/Users/daanwichmann/PycharmProjects/Explainable AI/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


### Adding custom classification head

In [12]:
model.fc = nn.Linear(in_features=2048, out_features=2)  # Got the input features from the sourcecode hihi

In [13]:
model = model.to(device)

### Freezing layers

## Training the model

### Setting hyper parameters

In [14]:
EPOCHS = 25
LR = 1e-4

### Initializing optimizer and loss function

In [15]:
optimizer = Adam(params=model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss().to(device)

### Setting model to train mode and moving to device

In [14]:
model.train()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [ ]:
for epoch in range(EPOCHS):
    running_loss = 0

    for X, y in tqdm(train_loader, f"Epoch {epoch +1}"):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)[0]
        loss = criterion(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch complete: {epoch + 1}/{EPOCHS} (loss: {running_loss})")

## Evaluating the model

### Setting the model to evaluation mode

In [16]:
model.eval()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [17]:
correct = 0
total = len(test_dataset)

In [28]:
for X, y in test_loader:

    X, y = X.to(device), y.to(device)

    with torch.no_grad():
        y_pred = model(X)[0]
        y_pred = torch.argmax(y_pred, dim=-1)

    correct += torch.sum(y_pred == y)

In [29]:
acc = correct / total
print(f"Accuracy: {acc}")

Accuracy: 1.899999976158142
